# - N.B.: 
L'efficienza al 90% (WP nella sezione $\tau$ ID) è calcolata sulla base del VBF per avere una soglia comune, nel caso in cui voglia unire i due sample e trattarli come uno solo.

# Import

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import math
import seaborn as sns
import pickle
import copy

import torch
from torch import nn, Tensor

import time
import joblib

from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import fm.preprocessing as preProcess

In [4]:
# for the plots :
import mplhep as hep

plt.style.use(hep.style.CMS)

plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 13,

    "lines.linewidth": 2,

    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",

    "legend.frameon": False,
})


# Load the Datasets

In [5]:
VBF = np.load("VBFtoXto2Tau_M-30to300.npz")
ggF = np.load("GluGlutoXto2Tau_M-30to300.npz")

In [6]:
vbf = preProcess.extract_blocks(VBF)
ggf = preProcess.extract_blocks(ggF)

print("VBF N:", vbf["N"])
print("ggF N:", ggf["N"])

VBF N: 632014
ggF N: 1581413


# Build the DataFrames

In [7]:
tau_features = [
    "logpt","eta","phi","mass","dxy","dz",
    "ptCorrPNet","rawPNetVSjet","rawDeepTau2018v2p5VSjet",
    "charge",
    "dM_0","dM_1","dM_2","dM_10","dM_11",
    "leadTkDeltaEta","leadTkDeltaPhi","leadTkPtOverTauPt"
]

gen_tau_features = ["pt", "eta", "phi", "mass"]

met_features = [
    "MET_logpt","MET_phi","MET_covXX","MET_covXY",
    "MET_covYY","MET_significance","MET_sumEt","MET_sumPtUnclustered"
]

jet_features = ["logpt","eta","phi","mass"]

In [12]:
df_VBF = preProcess.build_df_from_npz(VBF, sample="VBF")
df_ggF = preProcess.build_df_from_npz(ggF, sample="ggF")

print("VBF:", df_VBF.shape)
print("ggF:", df_ggF.shape)

VBF: (632014, 69)
ggF: (1581413, 69)


In [13]:
# Aggiungo colonne utili e tolgo le correzioni PNet 'sbagliate':
df_VBF = preProcess.add_pt_and_targets(df_VBF)
df_ggF = preProcess.add_pt_and_targets(df_ggF)

print("VBF:", df_VBF.shape)
print("ggF:", df_ggF.shape)

VBF: (632012, 75)
ggF: (1581398, 75)


# Pre-Processing

## pT cut

In [14]:
pt_cfg = preProcess.PtCutConfig(
    pt_min=30.0,
    space="pt",
    tau1_pt_col="tau1_pt_reco",
    tau2_pt_col="tau2_pt_reco",
)

df_sel_VBF = preProcess.apply_pt_cut(df_VBF, pt_cfg)
df_sel_ggF = preProcess.apply_pt_cut(df_ggF, pt_cfg)

## $\tau$ ID

In [15]:
tauid_cfg = preProcess.TauIDConfig(
    tau1_id_col="tau1_rawPNetVSjet",
    tau2_id_col="tau2_rawPNetVSjet",
    invalid_value=-1.0,
    wp_mode="target_eff",
    target_eff=0.90,
    require_both=True,
)

In [16]:
df_VBF_valid = preProcess.filter_valid_tauid(df_sel_VBF, tauid_cfg)
df_ggF_valid = preProcess.filter_valid_tauid(df_sel_ggF, tauid_cfg)

print("VBF  after valid tauID:", len(df_sel_VBF), "->", len(df_VBF_valid))
print("ggF  after valid tauID:", len(df_sel_ggF), "->", len(df_ggF_valid))

VBF  after valid tauID: 394046 -> 391776
ggF  after valid tauID: 905844 -> 900532


In [17]:
thr1 = preProcess.threshold_for_target_eff(
    df_VBF_valid["tau1_rawPNetVSjet"].to_numpy(),
    tauid_cfg.target_eff
)
thr2 = preProcess.threshold_for_target_eff(
    df_VBF_valid["tau2_rawPNetVSjet"].to_numpy(),
    tauid_cfg.target_eff
)

print("Common thresholds from VBF:")
print("thr_tau1 =", thr1)
print("thr_tau2 =", thr2)

Common thresholds from VBF:
thr_tau1 = 0.81396484375
thr_tau2 = 0.60009765625


In [18]:
df_VBF_tauid = preProcess.apply_tauid_wp(df_VBF_valid, tauid_cfg, thr1, thr2)
df_ggF_tauid = preProcess.apply_tauid_wp(df_ggF_valid, tauid_cfg, thr1, thr2)

print("VBF after WP:", len(df_VBF_valid), "->", len(df_VBF_tauid))
print("ggF after WP:", len(df_ggF_valid), "->", len(df_ggF_tauid))

VBF after WP: 391776 -> 318448
ggF after WP: 900532 -> 705035


In [19]:
# check efficienze :
eff_vbf = len(df_VBF_tauid) / len(df_VBF_valid)
eff_ggf = len(df_ggF_tauid) / len(df_ggF_valid)

print("Event efficiency VBF:", eff_vbf)
print("Event efficiency ggF:", eff_ggf)

Event efficiency VBF: 0.8128318222657845
Event efficiency ggF: 0.7829094357557532


## Mass cut

In [20]:
print("VBF before cut:", len(df_VBF_tauid))

df_VBF_cut = df_VBF_tauid[
    (df_VBF_tauid["m_vis_ptcorr"] >= 80.0) &
    (df_VBF_tauid["m_vis_ptcorr"] <= 140.0)
].copy()

print("VBF after cut:", len(df_VBF_cut))

print("ggF before cut:", len(df_ggF_tauid))

df_ggF_cut = df_ggF_tauid[
    (df_ggF_tauid["m_vis_ptcorr"] >= 80.0) &
    (df_ggF_tauid["m_vis_ptcorr"] <= 140.0)
].copy()

print("VBF after cut:", len(df_ggF_cut))

VBF before cut: 318448
VBF after cut: 124101
ggF before cut: 705035
VBF after cut: 319372


# Save the valid DataFrames 

In [21]:
df_VBF_cut.to_pickle(os.path.join("df_VBF.pkl"))
df_ggF_cut.to_pickle(os.path.join("df_ggF.pkl"))

In [22]:
# Per ora non lo salvo - non ho abbastanza spazio libero
# se serve ci ritorno 
df_flat_all = pd.concat([df_VBF_cut, df_ggF_cut], axis=0).reset_index(drop=True)
df_flat_all.to_pickle(os.path.join("df_flat_all.pkl"))

In [55]:
import shutil
total, used, free = shutil.disk_usage("/")
print("Free space (GB):", free / 1e9)

Free space (GB): 0.090161152
